# G3 — Colab 실규모 수렴 확인 (Step 1, 단일 파이프)

**목적:** CLAUDE.md §13 Lock `G3` 해제용 실측. Step 1(기본 경로) 을 GNARL + MaskablePPO 로
학습해 **Colab T4 에서 수렴하는지**를 확인하고, 동시에 §9 V4-3(맨해튼 vs 복합 Φ 수렴속도) 비교를 위한
실행 틀을 제공한다.

**이 노트북은 CLAUDE.md 만 보고 새로 작성했다 (기존 코드 미참조).**

---

## CLAUDE.md 요구사항 ↔ 구현 위치

| CLAUDE.md | 요구 | 구현 |
|---|---|---|
| §2 Layer 0 | 하드제약은 mask 로만, 보상 금지 | `action_mask_27()` — 충돌/경계/역방향/벤딩반경 전부 mask |
| §2 Layer 1 | J = 단일 물리단위(kg), 상수 | `PipeSpec`, `step_J()` — 탐색 대상 아님 |
| §2 Layer 2 | r = -ΔJ + [γΦ(s') - Φ(s)] | `PipeRoutingEnvStep1.step()` |
| §3 | JIS 물리상수 (100A 기준) | `JIS_PIPE`, `JIS_FITTING` |
| §4 | 26방향 + STAY = 27, OPP26 역방향 금지 | `DIRS26`, `OPP26`, `MOVE_DIST` |
| §5 | 벤딩반경 mask + lookahead k (100A → k=6) | `action_mask_27()`, `K_TABLE` |
| §6 | GNARL feature extractor / J 정규화 / admissibility 클리핑 | `GNARLFeaturesExtractor` |
| §8 | 관측 벡터 50차원 | `build_obs()` |
| §9 | Φ = 맨해튼, Φ(종단)=0, γ_PBS=γ_PPO | `Phi_manhattan()`, `assert` |
| §11 | 환경 버그 체크리스트 6개 | `run_checklist_tests()` (셀 12) |

## 실행 순서
1. 셀 2 (설치) → 런타임 재시작 불필요
2. 셀 3~13 순차 실행 (시나리오 생성 A* 가 수 분 소요)
3. 셀 19 학습 (T4 기준 400k step ≈ 40~70분)
4. 셀 20~21 평가/판정

## 주의 (구현 중 발견된 CLAUDE.md 미기재/충돌 항목)
- `§6 admissibility 클리핑`: 26-연결 격자에서 **맨해튼은 h\* 의 상한(과대추정)** 이라
  `min(h_learned, h_manhattan)` 은 "폭주 방지 캡"이지 admissibility 보장이 아니다.
  기본값은 CLAUDE.md 그대로 `manhattan` 을 쓰고, 엄밀 하한(`octile3d`)은 옵션으로 제공한다. → **미결: G3-N1**
- `턴 각도 상한`: CLAUDE.md 는 180°(역방향)만 mask 한다. 120°/135° 급선회는 배관상 비현실적이라
  `max_turn_deg=90` 을 기본으로 두되 `180` 으로 끄면 CLAUDE.md 원문과 동일해진다. → **미결: G3-N2**
- `종단 처리`: PBS 불변조건(Φ(종단)=0)은 **목표 도달 종단에만** 적용해야 한다.
  데드락/타임아웃에 Φ=0 을 쓰면 `-Φ(s)>0` 이 되어 실패에 보너스가 붙는다(§9 probe4 위반).
  → 목표 도달만 `terminated`, 나머지는 `truncated` (부트스트랩). → **미결: G3-N3**

## 1. 설치

In [ ]:
#@title 1. 의존성 설치 (Colab)
# sb3-contrib = MaskablePPO (CLAUDE.md §0 변경불가: 자체 PPO 구현 금지)
!pip -q install "stable-baselines3>=2.3.0" "sb3-contrib>=2.3.0" "gymnasium>=0.29" wandb tqdm rich tensorboard
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 2. 설정 (G3Config)

환경 20×20×10 = 1.0m × 1.0m × 0.5m @ CELL 50mm. 파이프는 100A (k=6) 기준.

In [ ]:
#@title 2. import + 설정
import os, math, json, time, pickle, random, heapq
from dataclasses import dataclass, asdict, field
from typing import Optional, List, Dict, Tuple

import numpy as np

SEED = 20260919

@dataclass
class G3Config:
    # ---- 격자 (요구사항: 20x20x10 @ 50mm) ----
    NX: int = 20
    NY: int = 20
    NZ: int = 10
    CELL_M: float = 0.05

    # ---- 파이프 (CLAUDE.md §3) ----
    nominal: str = "100A"          # 100A 기준 시작 → k=6

    # ---- Layer 0 옵션 ----
    max_turn_deg: float = 180.0    # CLAUDE.md 원문(역방향만 금지). 90.0 으로 조이면 데드락 급증 — 셀 19 실측
    s0_full: bool = True           # 시작 노즐 직관 확보 가정 → s0=k (False 면 s0=0)

    # ---- Layer 2 (PBS, §9) ----
    phi_type: str = "manhattan"    # "manhattan" | "combined"  (V4-3 비교용)
    phi_goal_ratio: float = 0.8    # combined 일 때 goal 비중
    phi_cong_ratio: float = 0.2    # combined 일 때 혼잡도 비중
    gamma: float = 0.99            # γ_PPO == γ_PBS (§9 조건 2)

    # ---- 에피소드 ----
    max_steps: int = 200
    invalid_penalty: float = 0.01  # §11 체크리스트: 무효행동 = 제자리 + 페널티 (마스킹 하에선 발생 안 함)
    max_invalid: int = 5
    deadlock_1ply: bool = False    # Layer 0 확장(선택): 다음 상태가 데드락인 이동도 마스킹. 셀 13 이 효과를 잰다
    fail_cost_mode: str = "const"  # 데드락 종단 비용: "const" | "h_star"(남은 최소 J) | "none". 셀 8 참조
    fail_cost_const: float = 2.0   # 정규화 J 기준 어떤 완주보다도 비싸게 (J_MAX=DIAG_J 이므로 완주는 ~1.0)

    # ---- 관측 (§8) ----
    ray_max: int = 6               # SDF 센서 최대 사거리(칸)

    # ---- 시나리오 ----
    n_train_scen: int = 192
    n_eval_scen: int = 48
    max_start_dirs: int = 2        # 시나리오당 초기방향 후보 수 (§11 V3-D)
    obstacle_fill: float = 0.12
    min_path_cells: int = 12

    # ---- GNARL (§6) ----
    features_dim: int = 64         # = 일반특징 63 + h(n) 1
    hidden: int = 128
    h_clip_mode: str = "manhattan" # "manhattan"(CLAUDE.md §6) | "octile3d"(엄밀 하한) | "none"
    pretrain_h: bool = True
    pretrain_epochs: int = 30

    # ---- PPO ----
    n_envs: int = 8
    n_steps: int = 512
    batch_size: int = 512
    n_epochs: int = 10
    learning_rate: float = 3e-4
    ent_coef: float = 0.05         # 데드락 지형 탈출에 필요 (셀 19 실측)
    clip_range: float = 0.2
    total_timesteps: int = 2_000_000   # CPU 8envs 실측 42분. 400k 로는 부족하다 (셀 19 실측)
    eval_every: int = 50_000

    # ---- wandb ----
    wandb_project: str = "pipe-routing-gnarl"
    wandb_run_name: str = ""
    wandb_mode: str = ""           # "" 면 자동 (API key 있으면 online, 없으면 offline)

    # ---- G3 판정 기준 (추측: 목표치) ----
    target_success_rate: float = 0.95
    target_length_ratio: float = 1.10

cfg = G3Config()
print(json.dumps(asdict(cfg), ensure_ascii=False, indent=2))

## 3. Layer 1 — JIS 물리 상수 (CLAUDE.md §3)

**탐색 대상이 아니다 (V2 확정).** 규격/단가표에서 조회한 상수이며 autoresearch 는 여기를 건드리지 않는다.
스케줄 Sch40 고정, 밸브 게이트밸브만, 용접/플랜지 무시.

In [ ]:
#@title 3. JIS 상수 + PipeSpec + J 정규화
# CLAUDE.md §3 직관 중량 (JIS G3454 Sch40)
JIS_PIPE = {   # 호칭경: (OD mm, kg/m)
    "15A": (21.7, 1.27), "25A": (34.0, 2.57),  "50A": (60.5, 5.44),
    "65A": (76.3, 9.11), "100A": (114.3, 16.10), "150A": (165.2, 28.20),
    "200A": (216.3, 42.50), "300A": (318.5, 79.70), "400A": (406.4, 124.00),
    "500A": (508.0, 185.00),
}
# CLAUDE.md §3 관이음 중량 (JIS B2312, LR) — 표에 있는 구경만
JIS_FITTING = {  # 호칭경: (elbow90 kg, elbow45 kg, tee kg, gate valve kg, B/L m)
    "25A": (0.30, 0.17, 0.54, 2.0, 0.12),
    "65A": (1.80, 0.99, 3.24, 10.0, 0.20),
    "100A": (3.80, 2.09, 6.84, 20.0, 0.24),
    "200A": (18.0, 9.90, 32.4, 80.0, 0.42),
    "300A": (51.0, 28.1, 91.8, 210.0, 0.64),
    "500A": (195.0, 107.0, 351.0, 730.0, 1.05),
}
# CLAUDE.md §5 k값 (LR 90°, CELL=50mm 기준)
K_TABLE = {"15A": 2, "25A": 2, "50A": 3, "65A": 3, "100A": 6,
           "150A": 9, "200A": 12, "300A": 18, "500A": 30}

@dataclass(frozen=True)
class PipeSpec:
    nominal: str
    od_mm: float
    kg_per_m: float
    elbow90_kg: float
    elbow45_kg: float
    tee_kg: float
    valve_kg: float
    bl_m: float
    k: int

def make_spec(nominal: str) -> PipeSpec:
    od, kgm = JIS_PIPE[nominal]
    if nominal not in JIS_FITTING:
        raise KeyError(f"{nominal} 관이음 중량이 CLAUDE.md §3 표에 없음 (보간 금지: 규격표 조회 필요)")
    e90, e45, tee, val, bl = JIS_FITTING[nominal]
    return PipeSpec(nominal, od, kgm, e90, e45, tee, val, bl, K_TABLE[nominal])

SPEC = make_spec(cfg.nominal)
K = SPEC.k                      # 100A → 6 (요구사항)
PAD = max(cfg.ray_max, K + 1) + 1   # 패딩 폭: lookahead 최대 오프셋(1+k), ray 사거리 커버

# CLAUDE.md §6 — J 정규화 (미적용 시 h(n) 학습 실패, G1-Q2)
DIAG_J = math.sqrt(cfg.NX**2 + cfg.NY**2 + cfg.NZ**2) * cfg.CELL_M * SPEC.kg_per_m
J_MAX = DIAG_J                  # §9 Φ 정규화도 동일 스케일 사용
SCALE = 1.0 / DIAG_J

print(SPEC)
print(f"k = {K} 칸 (필요 직진 {K*cfg.CELL_M*1000:.0f} mm), PAD = {PAD}")
print(f"DIAG_J = {DIAG_J:.3f} kg | 직진 1칸 = {cfg.CELL_M*SPEC.kg_per_m:.4f} kg "
      f"| 90도 엘보 = {SPEC.elbow90_kg:.2f} kg (= 직진 {SPEC.elbow90_kg/(cfg.CELL_M*SPEC.kg_per_m):.1f}칸)")

## 4. 행동 공간 (CLAUDE.md §4)

26방향 + STAY = **27**. 역방향은 `OPP26` 으로 매 스텝 1개 마스킹.
이동거리는 면=1.0 / 모서리=√2 / 꼭짓점=√3 보정 (V3-B).

`STAY` 는 **유효 이동이 하나도 없을 때(데드락)에만** 열린다.
평상시 열어두면 무한 제자리 루프가 생기고, 완전히 막으면 MaskablePPO 가 요구하는
"최소 1개 유효 행동" 조건이 깨진다.

In [ ]:
#@title 4. DIRS26 / OPP26 / MOVE_DIST / 턴 각도
DIRS26 = [
    (dx, dy, dz)
    for dx in (-1, 0, 1) for dy in (-1, 0, 1) for dz in (-1, 0, 1)
    if not (dx == 0 and dy == 0 and dz == 0)
]
assert len(DIRS26) == 26
DIR_TO_IDX = {d: i for i, d in enumerate(DIRS26)}
OPP26 = {i: DIR_TO_IDX[(-d[0], -d[1], -d[2])] for i, d in enumerate(DIRS26)}
assert all(OPP26[OPP26[i]] == i for i in range(26))     # V3-A: 26개 모두 정확

STAY = 26
N_ACTIONS = 27

MOVE_DIST = np.array([math.sqrt(dx*dx + dy*dy + dz*dz) for (dx, dy, dz) in DIRS26], dtype=np.float64)
DIRS_ARR = np.array(DIRS26, dtype=np.int64)             # (26,3)

def step_cost_kg(dir_idx: int, kg_per_m: float, cell_m: float) -> float:
    return float(MOVE_DIST[dir_idx]) * cell_m * kg_per_m

# 방향 i -> j 의 사잇각(도). 가능한 값: 0, 45, 54.7, 60, 70.5, 90, 109.5, 120, 125.3, 135, 180
_unit = DIRS_ARR / np.linalg.norm(DIRS_ARR, axis=1, keepdims=True)
TURN_DEG = np.degrees(np.arccos(np.clip(_unit @ _unit.T, -1.0, 1.0)))   # (26,26)

# 엘보 종류 판정 (JIS B2312 는 90°/45° 두 종류뿐이다).
# 26방향 격자에서 실제로 나오는 편향각: 45, 54.7, 60, 70.5, 90, 109.5, 120, 125.3, 135.
#   θ ≤ 67.5   → 45° 엘보 1개
#   θ ≤ 112.5  → 90° 엘보 1개
#   θ >  112.5 → 90° + 45° (엘보 하나로 못 꺾는다. 급선회를 J 에서 제대로 청구한다)
ELBOW45_MAX_DEG = 67.5
ELBOW90_MAX_DEG = 112.5
def elbow_kg_for(turn_deg: float, spec: PipeSpec) -> float:
    if turn_deg <= 1e-9:
        return 0.0
    if turn_deg <= ELBOW45_MAX_DEG:
        return spec.elbow45_kg
    if turn_deg <= ELBOW90_MAX_DEG:
        return spec.elbow90_kg
    return spec.elbow90_kg + spec.elbow45_kg

# lookahead 오프셋: 방향 i 로 꺾었을 때 검사할 칸들 (턴 목적지 기준 t=1..k)
LOOK_OFFS = np.stack([np.stack([DIRS_ARR[i] * t for t in range(1, K + 1)]) for i in range(26)])  # (26,k,3)

# 26방향 자유 직진 길이 채널 (obs[50:76], §8 확장) — 오프셋 t=1..k+1
RUN_LEN_CAP = K + 1
RUN_OFFS = np.stack([np.stack([DIRS_ARR[i] * t for t in range(1, RUN_LEN_CAP + 1)])
                     for i in range(26)])                # (26, k+1, 3)

# SDF 센서 10개 (§8 obs[40:50]) — 6면 + 수평 4대각
RAY_DIRS = np.array([(1,0,0), (-1,0,0), (0,1,0), (0,-1,0), (0,0,1), (0,0,-1),
                     (1,1,0), (1,-1,0), (-1,1,0), (-1,-1,0)], dtype=np.int64)
RAY_OFFS = np.stack([np.stack([RAY_DIRS[i] * t for t in range(1, cfg.ray_max + 1)])
                     for i in range(len(RAY_DIRS))])      # (10, ray_max, 3)
FACE_DIRS = np.array([(1,0,0), (-1,0,0), (0,1,0), (0,-1,0), (0,0,1), (0,0,-1)], dtype=np.int64)

print("행동 수:", N_ACTIONS, "| 유효 사잇각:", sorted(set(np.round(TURN_DEG.ravel(), 1))))

## 5. Layer 0 — 마스킹 (CLAUDE.md §2, §5)

하드 제약 4종을 **전부 `action_masks()` 에서만** 처리한다. 보상에 절대 넣지 않는다.

1. 충돌(장애물) / 2. 경계 위반 / 3. 역방향(`OPP26`) / 4. 벤딩 반경(직진칸수 s ≥ k **＋ lookahead k**)

lookahead 없으면 도달가능 상태의 2% 에서 데드락 (V1 실측). 사전계산은 §4.3 24MB 초과라 **on-demand**.
여기서는 numpy fancy-indexing 으로 26방향 lookahead 를 한 번에 조회한다 (파이썬 루프 제거).

**선택 옵션 `cfg.deadlock_1ply`** (기본 off): 이동 후 상태에서 유효 이동이 0개가 되는 행동까지 잘라낸다.
V1 의 lookahead 는 "꺾을 때 k칸"만 보므로, **직진으로 벽에 다가가 s<k 상태로 갇히는** 경우는 남는다.
보상이 아니라 mask 로 처리하므로 Layer 0 규칙에 어긋나지 않는다. 다만 매 스텝 26회 추가 마스크 계산이라
느리고, CLAUDE.md 에 없는 제약이므로 **기본 off + 셀 13 에서 효과만 측정**한다. → **미결: G3-N4**

> 이 노트북 작성 중 소규모 실측(60 에피소드, 랜덤 정책): 데드락율 **0.80 → 0.78**, 마스크가 실제로 달라진
> 스텝은 **1.4%** 뿐이었다. 즉 데드락은 대부분 **2수 이상 앞**에서 결정되므로 1-ply 로는 거의 못 막는다.
> 기본 off 인 이유다. (k칸 앞까지 보는 n-ply 로 키우면 비용이 26^n 으로 폭발 — 하지 말 것.)

**목표칸 예외:** lookahead 구간 안에 목표칸이 들어오면 그 방향은 통과시킨다.
(목표는 노즐 종단이라 그 뒤 직관을 요구할 수 없다. 이 예외가 없으면 목표 주변이 통째로 도달 불가가 된다.)
이 규칙은 A* 기준선과 환경이 **같은 함수**를 쓰므로 둘의 제약이 어긋날 수 없다.

### 관측 벡터 (CLAUDE.md §8) — 26채널 확장

§8 의 50차원을 그대로 두고 **뒤에 26채널을 붙였다** (§8 이 규정한 확장 방식).

```
obs[ 0: 3]  현재 위치 xyz (정규화)        obs[34:40]  이웃 6면 SDF (자유 +1 / 막힘 -1)
obs[ 3: 6]  목표 위치 xyz (정규화)        obs[40:50]  SDF 센서 10개 (6면 + 수평 4대각)
obs[ 6:32]  현재 방향 one-hot (26)        obs[50:76]  ← 추가: 26방향 자유 직진 길이 / (k+1)
obs[32]     직진 칸수 min(s,k)/k
obs[33]     g_cost 누적 / J_MAX
```

**왜 추가했나 (실측):** §8 의 SDF 센서 10개는 6면 + 수평 4대각뿐이라
`(1,1,1)` 같은 **꼭짓점 방향으로 진행 중이면 정면 장애물이 관측에 전혀 안 잡힌다.**
k=6 마스킹 하에서는 벽까지 5칸 남은 상태에서 이미 늦는다 (s<k 라 꺾지 못하고 데드락).
이 채널 없이 400k 스텝을 돌린 실측: **success 0.02~0.06 / deadlock 0.95 에서 정체**.
26방향 직진여유는 **마스크가 보는 것과 같은 정보**를 정책에도 주는 채널이다.

> Step 전이 시에는 `zero_pad_state_dict()` 로 입력 가중치를 0-padding 해 이어받는다 (§8 규칙).

In [ ]:
#@title 5. 격자 / 마스킹 / 관측 / Φ
DIMS = np.array([cfg.NX - 1, cfg.NY - 1, cfg.NZ - 1], dtype=np.float64)
OBS_DIM = 76      # §8 의 50 + 26방향 직진여유 채널 (아래 설명)

def make_pad(free: np.ndarray) -> np.ndarray:
    """경계 밖 = 막힘(False) 이 되도록 PAD 만큼 패딩. free3() 의 경계검사를 인덱싱으로 대체."""
    p = np.zeros((cfg.NX + 2*PAD, cfg.NY + 2*PAD, cfg.NZ + 2*PAD), dtype=bool)
    p[PAD:PAD+cfg.NX, PAD:PAD+cfg.NY, PAD:PAD+cfg.NZ] = free
    return p

def free3(pad: np.ndarray, x: int, y: int, z: int) -> bool:
    if abs(x) > cfg.NX + PAD or abs(y) > cfg.NY + PAD or abs(z) > cfg.NZ + PAD:
        return False
    return bool(pad[x + PAD, y + PAD, z + PAD])

def action_mask_27(pad, pos, dir_idx, s, goal, k=K, max_turn_deg=None,
                   deadlock_1ply=None) -> np.ndarray:
    """Layer 0 전부. 반환 (27,) bool. 환경과 A* 가 공유한다."""
    if max_turn_deg is None:
        max_turn_deg = cfg.max_turn_deg
    if deadlock_1ply is None:
        deadlock_1ply = cfg.deadlock_1ply
    m = np.zeros(N_ACTIONS, dtype=bool)

    # §11 체크리스트: 현재 위치가 경계 밖/막힘이면 전부 False
    px, py, pz = pos[0] + PAD, pos[1] + PAD, pos[2] + PAD
    if not (0 <= px < pad.shape[0] and 0 <= py < pad.shape[1] and 0 <= pz < pad.shape[2]):
        return m
    if not pad[px, py, pz]:
        return m

    pos_a = np.asarray(pos, dtype=np.int64)
    goal_a = np.asarray(goal, dtype=np.int64)
    nb = pos_a + DIRS_ARR                                    # (26,3) 이동 목적지
    nbp = nb + PAD
    cand = pad[nbp[:, 0], nbp[:, 1], nbp[:, 2]].copy()       # 충돌 + 경계

    cand[OPP26[dir_idx]] = False                             # 역방향 (V3-A)
    is_turn = np.arange(26) != dir_idx
    cand &= ~(is_turn & (TURN_DEG[dir_idx] > max_turn_deg))  # 급선회 상한 (미결 G3-N2)
    if s < k:
        cand &= ~is_turn                                     # 직진 칸수 부족 → 턴 금지

    need = np.where(cand & is_turn)[0]                       # lookahead 대상
    if need.size:
        cells = nb[need][:, None, :] + LOOK_OFFS[need]       # (n,k,3)
        cp = cells + PAD
        free_c = pad[cp[..., 0], cp[..., 1], cp[..., 2]]     # (n,k)
        blocked = ~free_c
        is_goal_c = (cells == goal_a).all(-1)                # (n,k)
        first_block = np.where(blocked.any(1), blocked.argmax(1), k)
        first_goal = np.where(is_goal_c.any(1), is_goal_c.argmax(1), k + 1)
        goal_is_next = (nb[need] == goal_a).all(1)
        ok = goal_is_next | (~blocked.any(1)) | (first_goal < first_block)
        cand[need[~ok]] = False

    # (선택) 1-ply 데드락 회피: 다음 상태에서 유효 이동이 0개인 행동도 잘라낸다.
    # 보상이 아니라 mask 이므로 Layer 0 규칙을 지킨다. 재귀 방지를 위해 내부 호출은 항상 off.
    if deadlock_1ply:
        for i in np.where(cand)[0]:
            i = int(i)
            npos = (pos_a[0] + DIRS_ARR[i, 0], pos_a[1] + DIRS_ARR[i, 1], pos_a[2] + DIRS_ARR[i, 2])
            if tuple(int(v) for v in npos) == tuple(int(v) for v in goal_a):
                continue
            ns = min(s + 1, k) if i == dir_idx else 1
            nxt = action_mask_27(pad, npos, i, ns, goal, k, max_turn_deg, deadlock_1ply=False)
            if not nxt[:26].any():
                cand[i] = False

    m[:26] = cand
    if not m.any():
        m[STAY] = True      # 데드락: MaskablePPO 는 유효 행동 ≥1 을 요구한다
    return m

# ---- Layer 1: J (kg) ----
def move_J(dir_idx: int, cur_dir: int, spec: PipeSpec) -> Tuple[float, float, int]:
    """한 칸 이동의 ΔJ. 반환 (ΔJ kg, 이동거리 m, 엘보종류 0=없음/45/90)."""
    dist_m = float(MOVE_DIST[dir_idx]) * cfg.CELL_M
    dJ = dist_m * spec.kg_per_m
    kind = 0
    if dir_idx != cur_dir:
        deg = float(TURN_DEG[cur_dir, dir_idx])
        dJ += elbow_kg_for(deg, spec)
        kind = 45 if deg <= ELBOW45_MAX_DEG else 90
        if deg > ELBOW90_MAX_DEG:
            kind = 135                        # 90 + 45 (아래에서 둘 다 집계)
    return dJ, dist_m, kind

# ---- 관측 (§8) ----
def build_obs(pad, pos, dir_idx, s, g_J, goal, k=K) -> np.ndarray:
    o = np.zeros(OBS_DIM, dtype=np.float32)
    pos_a = np.asarray(pos, dtype=np.float64)
    goal_a = np.asarray(goal, dtype=np.float64)
    o[0:3] = pos_a / DIMS                       # 현재 위치 xyz
    o[3:6] = goal_a / DIMS                      # 목표 위치 xyz
    o[6 + dir_idx] = 1.0                        # 현재 방향 one-hot (26)
    o[32] = min(s, k) / k                       # 직진 칸수
    o[33] = min(g_J / J_MAX, 1.0)               # g_cost 누적 (정규화)
    ip = np.asarray(pos, dtype=np.int64)
    f = ip + FACE_DIRS + PAD                    # 이웃 6면 SDF
    o[34:40] = np.where(pad[f[:, 0], f[:, 1], f[:, 2]], 1.0, -1.0)
    rc = ip + RAY_OFFS + PAD                    # SDF 센서 10개
    rf = pad[rc[..., 0], rc[..., 1], rc[..., 2]]        # (10, ray_max)
    o[40:50] = np.cumprod(rf, axis=1).sum(axis=1) / cfg.ray_max
    # obs[50:76] — 26방향 자유 직진 길이 (k+1 칸에서 포화). §8 확장 채널.
    # 이게 없으면 에이전트는 '자기 진행방향의 벽'을 못 본다 (§8 아래 설명 참조).
    uc = ip + RUN_OFFS + PAD
    uf = pad[uc[..., 0], uc[..., 1], uc[..., 2]]        # (26, k+1)
    o[50:76] = np.cumprod(uf, axis=1).sum(axis=1) / RUN_LEN_CAP
    return o

def zero_pad_state_dict(sd: dict, new_in_dim: int, first_layer_keys=("feat_net.0.weight", "h_net.0.weight")):
    """§8: Step 전이 시 관측 채널이 뒤에 붙으므로 입력 가중치를 0-padding 해서 load 호환."""
    out = dict(sd)
    for kkey in first_layer_keys:
        if kkey in out:
            w = out[kkey]
            if w.shape[1] < new_in_dim:
                pad_w = w.new_zeros((w.shape[0], new_in_dim - w.shape[1]))
                out[kkey] = torch.cat([w, pad_w], dim=1)
    return out

# ---- Layer 2: Φ (§9) ----
def Phi(pos, goal, pad=None, spec=SPEC) -> float:
    """PBS 포텐셜. 조건: Φ(종단)=0, state-only, γ_PBS=γ_PPO."""
    if tuple(pos) == tuple(goal):
        return 0.0                              # ← §9 조건 1 (종단 명시)
    if cfg.phi_type == "manhattan":
        d = abs(pos[0]-goal[0]) + abs(pos[1]-goal[1]) + abs(pos[2]-goal[2])
        return -d * cfg.CELL_M * spec.kg_per_m / J_MAX
    elif cfg.phi_type == "combined":
        e = math.dist(pos, goal)
        goal_term = -e * cfg.CELL_M * spec.kg_per_m / J_MAX
        ip = np.asarray(pos, dtype=np.int64)
        rc = ip + RAY_OFFS + PAD
        sdf = float(np.cumprod(pad[rc[..., 0], rc[..., 1], rc[..., 2]], axis=1).sum(axis=1).min())
        cong_term = -1.0 / (sdf * 5.0 + 1.0)
        return cfg.phi_goal_ratio * goal_term + cfg.phi_cong_ratio * cong_term
    raise ValueError(cfg.phi_type)

assert Phi((3, 3, 3), (3, 3, 3)) == 0.0, "§9 조건 1 위반"
print("Φ(start~) 예시:", round(Phi((0,0,0), (19,19,9)), 4), "| Φ(종단) =", Phi((5,5,5), (5,5,5)))

## 6. G1 구조 — J 기반 A* (기준선 겸 h(n) 학습 라벨)

CLAUDE.md §6 G1: `g(n) = J_fn(...)`, `f = g + h`. **보상함수 변환 불필요.**

이 A* 는 세 가지 용도로 쓴다.
1. **시나리오 도달성 검증** — §5/§11: k 제약 하에서 도달 불가한 시나리오를 걸러낸다 (k=0 A* 로는 부족).
2. **평가 기준선** — `length_ratio = 에이전트 경로길이 / A* 경로길이`.
3. **h(n) 사전학습 라벨** — 최적경로의 접미(suffix)는 그 상태에서의 최적이므로 `J_total - g(n)` 이 정확한 cost-to-go.

휴리스틱은 26-연결 격자의 **엄밀 하한**(octile3d)을 쓴다. 맨해튼은 대각이동 때문에 과대추정이라
A* 최적성을 깨뜨린다 (§6 클리핑 관련 미결 G3-N1 과 같은 이유).

In [ ]:
#@title 6. J 기반 A* (k 제약 공유)
SQRT2, SQRT3 = math.sqrt(2.0), math.sqrt(3.0)

def h_octile3d_cells(pos, goal) -> float:
    """26-연결 격자에서 (pos→goal) 이동거리의 엄밀 하한 (칸 단위, 장애물 무시)."""
    d = sorted((abs(pos[0]-goal[0]), abs(pos[1]-goal[1]), abs(pos[2]-goal[2])), reverse=True)
    a, b, c = d
    return (a - b) * 1.0 + (b - c) * SQRT2 + c * SQRT3

# h_fn 규약: h_fn(pos, goal, dir_idx, s, pad) -> kg. 학습된 h(n) 도 같은 규약으로 끼운다.
def h_octile3d_J(pos, goal, dir_idx=0, s=0, pad=None, spec=SPEC) -> float:
    return h_octile3d_cells(pos, goal) * cfg.CELL_M * spec.kg_per_m

def h_manhattan_J(pos, goal, dir_idx=0, s=0, pad=None, spec=SPEC) -> float:
    d = abs(pos[0]-goal[0]) + abs(pos[1]-goal[1]) + abs(pos[2]-goal[2])
    return d * cfg.CELL_M * spec.kg_per_m

def astar_gnarl(pad, start, goal, start_dir, h_fn=h_octile3d_J, spec=SPEC, k=K,
                s0=None, max_expand=300_000):
    """CLAUDE.md §6 G1: g(n)=J, h(n)=h_fn. 상태 = (pos, dir, min(s,k))."""
    if s0 is None:
        s0 = k if cfg.s0_full else 0
    start = tuple(int(v) for v in start); goal = tuple(int(v) for v in goal)
    s_start = (start, int(start_dir), min(int(s0), k))
    g = {s_start: 0.0}
    came = {}
    cnt = 0
    heap = [(h_fn(start, goal, int(start_dir), min(int(s0), k), pad), 0, s_start)]
    expanded = 0
    closed = set()
    while heap:
        f, _, st = heapq.heappop(heap)
        if st in closed:
            continue
        closed.add(st)
        pos, d, s = st
        if pos == goal:
            return _reconstruct(came, st, g, spec, expanded)
        expanded += 1
        if expanded > max_expand:
            return None
        m = action_mask_27(pad, pos, d, s, goal, k)
        for i in np.where(m[:26])[0]:
            i = int(i)
            dJ, _, _ = move_J(i, d, spec)
            npos = (pos[0]+DIRS26[i][0], pos[1]+DIRS26[i][1], pos[2]+DIRS26[i][2])
            ns = min(s + 1, k) if i == d else 1
            nst = (npos, i, ns)
            ng = g[st] + dJ
            if ng < g.get(nst, math.inf) - 1e-12:
                g[nst] = ng
                came[nst] = st
                cnt += 1
                heapq.heappush(heap, (ng + h_fn(npos, goal, i, ns, pad), cnt, nst))
    return None

def _reconstruct(came, st, g, spec, expanded):
    states = [st]
    while st in came:
        st = came[st]
        states.append(st)
    states.reverse()
    J = g[states[-1]]
    len_m, b90, b45 = 0.0, 0, 0
    for a, b in zip(states[:-1], states[1:]):
        i, dprev = b[1], a[1]
        len_m += float(MOVE_DIST[i]) * cfg.CELL_M
        if i != dprev:
            deg = float(TURN_DEG[dprev, i])
            if deg <= ELBOW45_MAX_DEG: b45 += 1
            elif deg <= ELBOW90_MAX_DEG: b90 += 1
            else: b90 += 1; b45 += 1          # 급선회 = 90 + 45
    return dict(J=J, len_m=len_m, n_cells=len(states) - 1, bends90=b90, bends45=b45,
                expanded=expanded,
                states=[(s[0], s[1], s[2], g[s]) for s in states])   # h(n) 라벨용

## 7. 시나리오 생성기 (§5, §11 체크리스트)

- 장애물 배치 후 **k 제약 하에서 A* 로 도달성 재검증** (k=0 A* 로는 100A 이상 8% 가 도달불가 — V1 발견)
- 초기 방향을 **하드코딩하지 않는다**: 시나리오마다 유효 초기방향 후보를 최대 `max_start_dirs` 개 뽑고,
  그중 **A* 로 풀리는 방향만** 남긴다 (V3-D 버그 방지). 학습 시 매 리셋마다 그중 하나를 랜덤 선택.
- 방향별로 최적 J/길이를 저장해 평가 기준선으로 쓴다.

In [ ]:
#@title 7. 시나리오 생성
from tqdm.auto import tqdm

def gen_obstacle_grid(rng: np.random.Generator) -> np.ndarray:
    free = np.ones((cfg.NX, cfg.NY, cfg.NZ), dtype=bool)
    target = int(cfg.obstacle_fill * free.size)
    filled = 0
    guard = 0
    while filled < target and guard < 1000:
        guard += 1
        sx, sy, sz = (int(rng.integers(2, 5)), int(rng.integers(2, 5)), int(rng.integers(1, 5)))
        x = int(rng.integers(0, cfg.NX - sx + 1)); y = int(rng.integers(0, cfg.NY - sy + 1))
        z = int(rng.integers(0, cfg.NZ - sz + 1))
        blk = free[x:x+sx, y:y+sy, z:z+sz]
        filled += int(blk.sum())
        free[x:x+sx, y:y+sy, z:z+sz] = False
    return free

def make_scenario(rng: np.random.Generator, max_tries: int = 60) -> Optional[dict]:
    for _ in range(max_tries):
        free = gen_obstacle_grid(rng)
        pad = make_pad(free)
        idx = np.argwhere(free)
        if len(idx) < 50:
            continue
        a = tuple(int(v) for v in idx[rng.integers(len(idx))])
        b = tuple(int(v) for v in idx[rng.integers(len(idx))])
        if sum(abs(x - y) for x, y in zip(a, b)) < cfg.min_path_cells:
            continue
        # 유효 초기 방향 후보 (첫 칸이 비어 있는 방향) → 셔플 후 최대 max_start_dirs 개
        cand = [i for i in range(26) if free3(pad, a[0]+DIRS26[i][0], a[1]+DIRS26[i][1], a[2]+DIRS26[i][2])]
        rng.shuffle(cand)
        dirs, sol = [], {}
        for d in cand[:max(1, cfg.max_start_dirs)]:
            r = astar_gnarl(pad, a, b, d)          # ← k 제약 하 도달성 검증
            if r is not None:
                dirs.append(int(d)); sol[int(d)] = r
        if not dirs:
            continue
        return dict(free=free, pad=pad, start=a, goal=b, dirs=dirs, astar=sol)
    return None

def build_pool(n: int, seed: int, desc: str) -> List[dict]:
    rng = np.random.default_rng(seed)
    pool = []
    pbar = tqdm(total=n, desc=desc)
    while len(pool) < n:
        sc = make_scenario(rng)
        if sc is not None:
            pool.append(sc); pbar.update(1)
    pbar.close()
    return pool

# 캐시 키에 시나리오를 바꾸는 설정을 전부 넣는다.
# (구경/격자/장애물/턴상한/k 를 바꿨는데 옛 캐시가 조용히 로드되면 전부 무의미한 실험이 된다)
CACHE = (f"g3_scen_{cfg.nominal}_{cfg.NX}x{cfg.NY}x{cfg.NZ}_k{K}_"
         f"n{cfg.n_train_scen}+{cfg.n_eval_scen}_d{cfg.max_start_dirs}_"
         f"f{cfg.obstacle_fill}_t{cfg.max_turn_deg:.0f}_m{cfg.min_path_cells}.pkl")
if os.path.exists(CACHE):
    with open(CACHE, "rb") as f:
        TRAIN_POOL, EVAL_POOL = pickle.load(f)
    print("캐시 로드:", CACHE)
else:
    t0 = time.time()
    TRAIN_POOL = build_pool(cfg.n_train_scen, SEED, "train scenarios")
    EVAL_POOL = build_pool(cfg.n_eval_scen, SEED + 1, "eval scenarios")
    with open(CACHE, "wb") as f:
        pickle.dump((TRAIN_POOL, EVAL_POOL), f)
    print(f"생성 완료 {time.time()-t0:.1f}s → {CACHE}")

_stat = [sc["astar"][sc["dirs"][0]] for sc in EVAL_POOL]
print(f"train={len(TRAIN_POOL)} eval={len(EVAL_POOL)}")
print(f"eval 기준선: J {np.mean([s['J'] for s in _stat]):.2f} kg | "
      f"길이 {np.mean([s['len_m'] for s in _stat]):.2f} m | "
      f"칸수 {np.mean([s['n_cells'] for s in _stat]):.1f} | "
      f"벤딩 {np.mean([s['bends90']+s['bends45'] for s in _stat]):.1f} | "
      f"A* 확장노드 {np.mean([s['expanded'] for s in _stat]):.0f}")

## 8. 환경 — Step 1 (CLAUDE.md §2 3층 구조)

```
Layer 0  action_masks()  ← 충돌/경계/역방향/벤딩반경. 보상에 절대 없음
Layer 1  J (kg)          ← 규격표 상수. 길이 + 엘보
Layer 2  r = -ΔJ/J_MAX + [γΦ(s') - Φ(s)]
```

### 종단 처리 (미결 G3-N3)

`Φ(종단)=0` (§9 조건 1) 은 **목표 도달 종단에만** 적용한다. `Phi()` 가 목표칸에서 0 을 돌려주므로 자동이다.
**데드락/타임아웃 상태에 Φ=0 을 쓰면 안 된다.** 쓰면 그 스텝의 shaping 이 `-Φ(s) > 0` 이 되어
실패에 보너스가 붙는다. 이 격자의 실제 수치로 확인되는 문제다:

| 경로 | 총 보상 |
|---|---|
| 1스텝 만에 자살(Φ=0 적용) | `-0.033 + (0 - Φ(s₀))` ≈ **+1.53** |
| 목표까지 정상 도달 | `-J/J_MAX - Φ(s₀)` ≈ -1.2 + 1.57 ≈ **+0.37** |

즉 Φ(종단)=0 을 실패 종단까지 확대하면 **즉시 자살이 최적 정책**이 된다 (§9 probe4 가 경고한 목적함수 변형).
그래서 데드락은 Φ 를 0 으로 치환하지 않고 실제 Φ(s') 를 그대로 쓴다 — 총 shaping 이
`-Φ(s₀) + γᵀΦ(s_T)` 가 되어 실패가 목표 도달보다 항상 불리하다.

종단 플래그는 이렇게 나눈다.

| 종료 사유 | 플래그 | 이유 |
|---|---|---|
| 목표 도달 | `terminated` | 진짜 종단. Φ=0 |
| 데드락 (유효 이동 0개) | `terminated` | 이어질 상태가 **실제로 없다**. `truncated` 로 두면 SB3 가 V(s') 로 부트스트랩해 허구의 미래가치가 붙고, 에이전트가 데드락을 선호할 수 있다 |
| 타임아웃 (`max_steps`) | `truncated` | 시간제한일 뿐 상태는 살아있다 → 부트스트랩이 맞다 |

### 데드락 종단 비용 `cfg.fail_cost_mode` (실측으로 추가된 항목)

위 설계만으로 120k 스텝을 돌려봤더니 **에이전트가 "빨리 포기하는 법"을 학습했다** (실측):

| timesteps | success | deadlock | 평균 스텝 |
|---|---|---|---|
| 15,000 | 0.042 | 0.875 | 51 |
| 30,000 | 0.042 | 0.917 | 42 |
| 45,000 | 0.000 | **1.000** | **35** |

원인: 한 스텝마다 `-ΔJ/J_MAX ≈ -0.033` 을 내는데 **종단하면 그 지출이 멈춘다**.
헤매다 실패(≈ -1.5)보다 **즉시 데드락(≈ -0.02)** 이 훨씬 이득이라 정책경사가 그쪽으로 간다.
목표 도달(≈ +0.4)이 최선이지만 찾기 어려워서, 쉬운 국소최적에 먼저 갇힌다.

해결: 데드락 종단에 비용을 물린다. "끊어진 배관은 남은 구간 값을 안 낸 게 아니다" 라는 뜻이라
**Layer 1 목적함수의 종단 조건**이지 Layer 0 벌점이 아니다 (하드 제약은 전부 mask 라 애초에 선택 불가).
성공 궤적끼리의 순위를 바꾸지 않으므로 최적 정책은 보존된다.

| `fail_cost_mode` | 종단 비용 | 8레이아웃 150k success (실측) |
|---|---|---|
| `none` | 0 | 0.125 |
| `h_star` | `h*(s_T)/J_MAX` (남은 최소 J) | 0.250 |
| **`const`** (기본) | `2.0` | **0.500** |

`h_star` 가 약한 이유: octile3d 는 엘보와 장애물 우회를 무시한 **하한**이라, 죽어도 남은 일을 깎아준다.
`const=2.0` 은 J_MAX = DIAG_J 라 완주 비용이 대략 1.0 인 점을 이용해 **어떤 완주보다도 비싸게** 만든 값이다.
→ **미결 G3-N3 에 포함**

**보상에 성공 보너스를 넣지 않는다.** 넣으면 Layer 1 의 J 가 목적함수가 아니게 된다.
PBS 텔레스코핑으로 목표 도달 시 총 shaping = `-Φ(s0) ≈ +1.7` 이라 이미 충분한 신호다.

In [ ]:
#@title 8. PipeRoutingEnvStep1
import gymnasium as gym
from gymnasium import spaces

def fail_cost_norm(pos, goal) -> float:
    """데드락 종단 비용. '미완성 경로는 남은 구간 값을 안 낸 게 아니다'.

    Layer 0 벌점이 아니다 (하드 제약은 전부 mask 로 처리되어 애초에 선택 불가).
    이것은 Layer 1 목적함수의 종단 조건이다: 목표에 닿지 못한 경로의 J 는
    '쓴 J + 남은 최소 J' 이고, 중간에 끊었다고 깎아주지 않는다.
    성공 궤적끼리의 순위는 건드리지 않으므로 최적 정책은 그대로다.
    """
    if cfg.fail_cost_mode == "none":
        return 0.0
    if cfg.fail_cost_mode == "const":
        return cfg.fail_cost_const
    return h_octile3d_J(pos, goal) / J_MAX          # 26-연결 엄밀 하한

class PipeRoutingEnvStep1(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self, pool: List[dict], spec: PipeSpec = SPEC, eval_mode: bool = False,
                 seed: Optional[int] = None):
        super().__init__()
        self.pool, self.spec, self.eval_mode = pool, spec, eval_mode
        self.k = spec.k
        self.rng = np.random.default_rng(seed)
        self.action_space = spaces.Discrete(N_ACTIONS)
        self.observation_space = spaces.Box(-1.0, 1.0, (OBS_DIM,), dtype=np.float32)
        self._cursor = 0
        self.sc = None

    # ---------- helpers ----------
    def _obs(self):
        return build_obs(self.pad, self.pos, self.dir, self.s, self.g_J, self.goal, self.k)

    def _info(self, success=False, deadlock=False, timeout=False):
        base = self.base
        ok = success and base is not None
        return dict(
            success=float(success),
            deadlock=float(deadlock),
            timeout=float(timeout),
            j_agent=float(self.g_J),
            j_ratio=float(self.g_J / base["J"]) if ok else float("nan"),
            length_ratio=float(self.len_m / base["len_m"]) if ok else float("nan"),
            n_bends=float(self.b90 + self.b45),
            bends_base=float(base["bends90"] + base["bends45"]) if base else float("nan"),
            n_invalid=float(self.n_invalid),
            ep_steps=float(self.t),
        )

    # ---------- gym API ----------
    def reset(self, *, seed=None, options=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        options = options or {}
        if "index" in options:
            i = int(options["index"]) % len(self.pool)
        elif self.eval_mode:
            i = self._cursor % len(self.pool); self._cursor += 1
        else:
            i = int(self.rng.integers(len(self.pool)))
        sc = self.pool[i]
        self.sc, self.pad = sc, sc["pad"]
        self.pos, self.goal = sc["start"], sc["goal"]

        # §11: 초기 방향 하드코딩 금지 — 유효 초기방향 집합에서 선택 (V3-D)
        if "dir" in options:
            self.dir = int(options["dir"])
        elif self.eval_mode:
            self.dir = int(sc["dirs"][0])                       # 평가는 결정적
        else:
            self.dir = int(self.rng.choice(sc["dirs"]))
        self.base = sc["astar"].get(self.dir)

        self.s = self.k if cfg.s0_full else 0                   # 시작 노즐 직관 가정
        self.g_J, self.len_m = 0.0, 0.0
        self.b90 = self.b45 = 0
        self.t, self.n_invalid = 0, 0
        self.path = [self.pos]
        return self._obs(), self._info()

    def action_masks(self) -> np.ndarray:
        # §11: 위치가 경계 밖이면 action_mask_27 이 전부 False 를 돌려준다
        return action_mask_27(self.pad, self.pos, self.dir, self.s, self.goal, self.k)

    def step(self, action: int):
        action = int(action)
        self.t += 1
        m = self.action_masks()
        reward = 0.0
        terminated = truncated = False

        # --- §11: 경계/유효성 검사. 무효 행동 = 제자리 유지 + 페널티 (G2-D 버그 원인) ---
        if action < 0 or action >= N_ACTIONS or not m[action]:
            self.n_invalid += 1
            reward -= cfg.invalid_penalty
            if self.n_invalid >= cfg.max_invalid or self.t >= cfg.max_steps:
                truncated = True
            return self._obs(), reward, terminated, truncated, self._info(timeout=truncated)

        if action == STAY:
            # 유효 이동이 0개일 때만 열리는 행동 = 데드락 (V1 의 2% 사례).
            # terminated=True: 이어질 상태가 실제로 없으므로 부트스트랩하면 허구의 미래가치가 붙는다.
            # Φ(데드락) 은 0 으로 만들지 않는다 (셀 8 설명 참조).
            reward -= fail_cost_norm(self.pos, self.goal)   # 미완성 잔여 J
            terminated = True
            return self._obs(), reward, terminated, truncated, self._info(deadlock=True)

        prev_pos = self.pos
        phi_prev = Phi(prev_pos, self.goal, self.pad, self.spec)

        dJ, dist_m, kind = move_J(action, self.dir, self.spec)          # Layer 1
        self.pos = (prev_pos[0] + DIRS26[action][0],
                    prev_pos[1] + DIRS26[action][1],
                    prev_pos[2] + DIRS26[action][2])
        self.s = min(self.s + 1, self.k) if action == self.dir else 1
        self.dir = action
        self.g_J += dJ
        self.len_m += dist_m
        if kind == 90: self.b90 += 1
        elif kind == 45: self.b45 += 1
        elif kind == 135: self.b90 += 1; self.b45 += 1
        self.path.append(self.pos)

        phi_new = Phi(self.pos, self.goal, self.pad, self.spec)          # Φ(목표)=0 보장
        reward += -dJ / J_MAX + cfg.gamma * phi_new - phi_prev           # Layer 2

        if self.pos == self.goal:
            terminated = True                                            # 진짜 종단
            return self._obs(), reward, terminated, truncated, self._info(success=True)
        if self.t >= cfg.max_steps:
            truncated = True                                             # 부트스트랩 대상
            return self._obs(), reward, terminated, truncated, self._info(timeout=True)
        return self._obs(), reward, terminated, truncated, self._info()

INFO_KEYS = ("success", "deadlock", "timeout", "j_ratio", "length_ratio", "n_bends", "ep_steps")

_e = PipeRoutingEnvStep1(EVAL_POOL, eval_mode=True)
_o, _i = _e.reset(options={"index": 0})
print("obs dim:", _o.shape, "| 유효 행동 수:", int(_e.action_masks().sum()),
      "| 기준선 J:", round(_e.base["J"], 2))

## 9. §11 환경 구현 체크리스트 검증

CLAUDE.md §11 은 V2·V3·G2 에서 실제로 나온 버그 패턴이다. 학습 전에 **전부 자동 검증**한다.
하나라도 실패하면 학습을 시작하지 않는다.

In [ ]:
#@title 9. 체크리스트 자동 검증 (실패 시 학습 금지)
def run_checklist_tests(pool=EVAL_POOL, n_ep=40, seed=7):
    rng = np.random.default_rng(seed)
    env = PipeRoutingEnvStep1(pool, eval_mode=False, seed=seed)
    results = {}

    # [1] step() 경계 검사 + 무효 행동 = 제자리 유지 + 페널티 (G2-D)
    env.reset(options={"index": 0})
    bad = int(np.where(~env.action_masks())[0][0])
    p0 = env.pos
    _, r, term, trunc, info = env.step(bad)
    results["[1] 무효행동=제자리+페널티"] = (env.pos == p0 and r < 0 and info["n_invalid"] == 1.0)

    # [2] action_masks(): 경계 밖 위치면 전부 False
    sc = pool[0]
    out = action_mask_27(sc["pad"], (-3, 5, 5), 0, K, sc["goal"])
    out2 = action_mask_27(sc["pad"], (cfg.NX + 2, 5, 5), 0, K, sc["goal"])
    results["[2] 경계밖 pos → 전부 False"] = (not out.any()) and (not out2.any())

    # [3] 초기 방향 하드코딩 금지 (V3-D)
    multi = [i for i, s in enumerate(pool) if len(s["dirs"]) > 1]
    if multi:
        seen = set()
        for _ in range(60):
            env.reset(options={"index": multi[0]}); seen.add(env.dir)
        results["[3] 초기방향 집합 사용"] = len(seen) > 1
    else:
        results["[3] 초기방향 집합 사용"] = True   # 후보가 1개뿐이면 검증 불가

    # [4] 시나리오 생성기: k 제약 하 도달성 (V1)
    results["[4] k 제약 도달성 검증됨"] = all(
        d in s["astar"] and s["astar"][d] is not None for s in pool for d in s["dirs"])

    # [5] J 정규화 상수 (G1-Q2)
    expect = math.sqrt(cfg.NX**2 + cfg.NY**2 + cfg.NZ**2) * cfg.CELL_M * SPEC.kg_per_m
    results["[5] DIAG_J 정규화"] = abs(DIAG_J - expect) < 1e-9 and abs(SCALE * DIAG_J - 1.0) < 1e-12

    # [6~9] 랜덤 마스킹 롤아웃 불변식
    rev_ok = turn_ok = pbs_ok = inside_ok = True
    invalid_seen = 0
    for ep in range(n_ep):
        obs, _ = env.reset()
        for _ in range(cfg.max_steps):
            m = env.action_masks()
            if m[OPP26[env.dir]]:
                rev_ok = False                                   # [6] 역방향 금지
            if env.s < env.k and m[:26].sum() > 0:
                allowed = np.where(m[:26])[0]
                if any(int(a) != env.dir for a in allowed):
                    turn_ok = False                              # [7] s<k 면 턴 불가
            a = int(rng.choice(np.where(m)[0]))
            prev_pos, prev_dir, prev_g = env.pos, env.dir, env.g_J
            phi_prev = Phi(prev_pos, env.goal, env.pad)
            obs, r, term, trunc, info = env.step(a)
            if a != STAY:
                dJ = env.g_J - prev_g
                exp_r = -dJ / J_MAX + cfg.gamma * Phi(env.pos, env.goal, env.pad) - phi_prev
                if abs(exp_r - r) > 1e-9:
                    pbs_ok = False                               # [8] r = -ΔJ + γΦ'-Φ
            if not (0 <= env.pos[0] < cfg.NX and 0 <= env.pos[1] < cfg.NY and 0 <= env.pos[2] < cfg.NZ):
                inside_ok = False                                # [9] 격자 이탈 없음
            invalid_seen += int(info["n_invalid"] > 0)
            if term or trunc:
                break
    results["[6] 역방향 상시 마스킹"] = rev_ok
    results["[7] s<k 면 턴 마스킹"] = turn_ok
    results["[8] r = -ΔJ + γΦ'-Φ"] = pbs_ok
    results["[9] 격자 이탈 없음"] = inside_ok
    results["[10] Φ(종단)=0"] = (Phi((4, 4, 4), (4, 4, 4)) == 0.0)
    return results

_res = run_checklist_tests()
for k_, v in _res.items():
    print(("  PASS  " if v else "  FAIL  ") + k_)
assert all(_res.values()), "§11 체크리스트 실패 — 학습 금지"
print("\n§11 체크리스트 전부 통과")

## 10. GNARL Feature Extractor (CLAUDE.md §6, G2)

**PPO 알고리즘 본체는 건드리지 않는다** (§0 변경불가: 자체 PPO 구현 금지).
`BaseFeaturesExtractor` 를 상속해 h(n) 신경망을 feature extractor **안쪽**에 넣는다.
rollout buffer / advantage / clip loss / action mask 는 sb3-contrib 원본 그대로 (G2 통과 4항목).

```
forward(obs) = concat[ feat_net(obs) (63) , h_admissible(obs) (1) ] = 64
```

### admissibility 클리핑 (미결 G3-N1)
CLAUDE.md §6 은 `min(h_learned, h_manhattan)` 을 규정한다. 다만 **26-연결 격자에서 맨해튼은
h\* 의 상한(과대추정)** 이므로 이 클리핑은 "폭주 방지 캡"이지 A* 최적성 보장이 아니다.
엄밀 하한은 octile3d 다. 기본값은 CLAUDE.md 원문(`manhattan`) 을 따르고, `cfg.h_clip_mode="octile3d"` 로
바꾸면 엄밀 admissible 이 되지만 학습된 h 가 하한에 눌려 정보량을 잃는다. **G3 에서 둘 다 측정할 것.**

In [ ]:
#@title 10. GNARLFeaturesExtractor
import torch
import torch.nn as nn
import torch.nn.functional as F
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

class GNARLFeaturesExtractor(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim: int = 64, hidden: int = 128,
                 clip_mode: str = "manhattan"):
        super().__init__(observation_space, features_dim)
        n_in = int(observation_space.shape[0])
        self.clip_mode = clip_mode
        self.feat_net = nn.Sequential(                       # 일반 특징 (features_dim - 1)
            nn.Linear(n_in, hidden), nn.ReLU(),
            nn.Linear(hidden, features_dim - 1), nn.ReLU(),
        )
        self.h_net = nn.Sequential(                          # h(n) 추정 (스칼라 1개)
            nn.Linear(n_in, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )
        self.register_buffer("dims_t", torch.tensor(DIMS, dtype=torch.float32))
        # 1칸당 J 를 J_MAX 로 정규화한 상수 (§6: 정규화 없으면 h 학습 실패)
        self.register_buffer("unit_t", torch.tensor(cfg.CELL_M * SPEC.kg_per_m / J_MAX,
                                                    dtype=torch.float32))

    def _delta_cells(self, obs: torch.Tensor) -> torch.Tensor:
        return (obs[:, 0:3] - obs[:, 3:6]).abs() * self.dims_t        # (B,3) 칸 단위

    def h_bound(self, obs: torch.Tensor) -> torch.Tensor:
        d = self._delta_cells(obs)
        if self.clip_mode == "octile3d":
            srt, _ = torch.sort(d, dim=-1, descending=True)
            a, b, c = srt[:, 0:1], srt[:, 1:2], srt[:, 2:3]
            cells = (a - b) + (b - c) * SQRT2 + c * SQRT3
        else:                                                          # "manhattan" (§6 원문)
            cells = d.sum(dim=-1, keepdim=True)
        return cells * self.unit_t

    def h_raw(self, obs: torch.Tensor) -> torch.Tensor:
        return F.softplus(self.h_net(obs))                             # h >= 0

    def h(self, obs: torch.Tensor) -> torch.Tensor:
        raw = self.h_raw(obs)
        if self.clip_mode == "none":
            return raw
        return torch.minimum(raw, self.h_bound(obs))                   # §6 안전 클리핑

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        return torch.cat([self.feat_net(obs), self.h(obs)], dim=-1)

# --- §11 체크리스트 [h(n) admissibility 클리핑] 검증 ---
_ext = GNARLFeaturesExtractor(_e.observation_space, cfg.features_dim, cfg.hidden, cfg.h_clip_mode)
_obs_batch = torch.as_tensor(np.stack([
    build_obs(sc["pad"], sc["start"], sc["dirs"][0], K, 0.0, sc["goal"]) for sc in EVAL_POOL]))
with torch.no_grad():
    _h, _b, _f = _ext.h(_obs_batch), _ext.h_bound(_obs_batch), _ext(_obs_batch)
assert bool((_h <= _b + 1e-6).all()), "admissibility 클리핑 실패"
assert _f.shape[1] == cfg.features_dim
print(f"클리핑 모드={cfg.h_clip_mode} | h<=bound 전부 통과 | features={_f.shape} "
      f"| bound 평균 {_b.mean():.3f} (정규화 J)")

## 11. h(n) 사전학습 (G1)

A* 최적경로의 **접미 최적성**을 이용해 정확한 cost-to-go 라벨을 만든다: `target = (J_total - g(n)) / J_MAX`.
PPO 손실에 보조항을 추가하지 않고 **학습 전에 따로** 맞춰 넣는다
(§0: PPO 본체 수정 금지 — 여기서는 feature extractor 가중치만 초기화하는 셈).

`cfg.pretrain_h=False` 로 끄고 학습하면 사전학습 기여도를 분리 측정할 수 있다.

In [ ]:
#@title 11. h(n) 사전학습 (A* 라벨)
def build_h_dataset(pool):
    X, Y = [], []
    for sc in pool:
        for d, sol in sc["astar"].items():
            Jt = sol["J"]
            for (pos, dd, ss, gg) in sol["states"]:
                X.append(build_obs(sc["pad"], pos, dd, ss, gg, sc["goal"]))
                Y.append((Jt - gg) / J_MAX)          # 남은 최적 J (정규화)
    return np.stack(X), np.asarray(Y, dtype=np.float32)[:, None]

def pretrain_h_net(extractor, X, Y, epochs=30, bs=256, lr=1e-3, val_frac=0.15, verbose=True):
    n = len(X); idx = np.random.default_rng(SEED).permutation(n)
    nv = int(n * val_frac)
    vi, ti = idx[:nv], idx[nv:]
    Xt = torch.as_tensor(X[ti]); Yt = torch.as_tensor(Y[ti])
    Xv = torch.as_tensor(X[vi]); Yv = torch.as_tensor(Y[vi])
    opt = torch.optim.Adam(extractor.h_net.parameters(), lr=lr)
    hist = []
    for ep in range(epochs):
        perm = torch.randperm(len(Xt))
        for i in range(0, len(Xt), bs):
            b = perm[i:i+bs]
            loss = F.mse_loss(extractor.h_raw(Xt[b]), Yt[b])
            opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():
            vr = extractor.h_raw(Xv)
            vmae = float((vr - Yv).abs().mean())
            clipped = float((vr > extractor.h_bound(Xv)).float().mean())
        hist.append((ep, vmae, clipped))
        if verbose and (ep % 5 == 0 or ep == epochs - 1):
            print(f"  ep {ep:3d} | val MAE {vmae:.4f} (= {vmae*J_MAX:.2f} kg) | 클리핑 발생률 {clipped:.1%}")
    return hist

Xh, Yh = build_h_dataset(TRAIN_POOL)
print(f"h(n) 데이터셋: {len(Xh)} 상태 | target 평균 {Yh.mean():.3f} (= {Yh.mean()*J_MAX:.2f} kg)")
H_EXTRACTOR_SD = None
if cfg.pretrain_h:
    _pre = GNARLFeaturesExtractor(_e.observation_space, cfg.features_dim, cfg.hidden, cfg.h_clip_mode)
    _hist = pretrain_h_net(_pre, Xh, Yh, epochs=cfg.pretrain_epochs)
    H_EXTRACTOR_SD = {k_: v.clone() for k_, v in _pre.h_net.state_dict().items()}
    # 기준선: 맨해튼/octile3d 휴리스틱의 같은 데이터셋 MAE
    with torch.no_grad():
        Xt_ = torch.as_tensor(Xh); Yt_ = torch.as_tensor(Yh)
        mae_net = float((_pre.h_raw(Xt_) - Yt_).abs().mean())
        _pre.clip_mode = "manhattan"; mae_man = float((_pre.h_bound(Xt_) - Yt_).abs().mean())
        _pre.clip_mode = "octile3d";  mae_oct = float((_pre.h_bound(Xt_) - Yt_).abs().mean())
        _pre.clip_mode = cfg.h_clip_mode
    print(f"\nMAE 비교 (kg): 학습 h {mae_net*J_MAX:.2f} | 맨해튼 {mae_man*J_MAX:.2f} | octile3d {mae_oct*J_MAX:.2f}")
else:
    print("사전학습 건너뜀 (cfg.pretrain_h=False)")

## 12. (선택) G1 speed_compare — 학습된 h(n) 을 A* 에 꽂기

§6 G1 의 원래 목적. 휴리스틱만 바꿔 **확장 노드 수**와 **해의 J** 를 비교한다.
`min(h_learned, h_manhattan)` 이 admissibility 를 보장하지 않으므로(미결 G3-N1)
J 가 octile3d 최적해보다 커지는지 함께 본다. 이 셀은 학습과 무관하므로 건너뛰어도 된다.

In [ ]:
#@title 12. (선택) A* 휴리스틱 비교
RUN_SPEED_COMPARE = True   #@param {type:"boolean"}

def make_learned_h(extractor, clip_mode=None):
    mode = clip_mode or extractor.clip_mode
    prev = extractor.clip_mode
    def h_fn(pos, goal, dir_idx=0, s=0, pad=None, spec=SPEC):
        o = torch.as_tensor(build_obs(pad, pos, dir_idx, s, 0.0, goal))[None, :]
        extractor.clip_mode = mode
        with torch.no_grad():
            v = float(extractor.h(o)[0, 0])
        extractor.clip_mode = prev
        return v * J_MAX
    return h_fn

if RUN_SPEED_COMPARE and H_EXTRACTOR_SD is not None:
    _cmp_ext = GNARLFeaturesExtractor(_e.observation_space, cfg.features_dim, cfg.hidden, cfg.h_clip_mode)
    _cmp_ext.h_net.load_state_dict(H_EXTRACTOR_SD)
    variants = {
        "octile3d (엄밀 하한)": h_octile3d_J,
        "manhattan (과대추정)": h_manhattan_J,
        f"학습 h(n) clip={cfg.h_clip_mode}": make_learned_h(_cmp_ext),
    }
    rows = {}
    for name, hf in variants.items():
        exp, jj, ok = [], [], 0
        for sc in EVAL_POOL[:8]:
            d = sc["dirs"][0]
            r = astar_gnarl(sc["pad"], sc["start"], sc["goal"], d, h_fn=hf)
            if r: exp.append(r["expanded"]); jj.append(r["J"]); ok += 1
        rows[name] = (np.mean(exp), np.mean(jj), ok)
    base_J = rows["octile3d (엄밀 하한)"][1]
    print(f"{'휴리스틱':28s} {'확장노드':>9s} {'J(kg)':>9s} {'최적대비':>9s}")
    for name, (e_, j_, ok) in rows.items():
        print(f"{name:28s} {e_:9.0f} {j_:9.2f} {j_/base_J:8.3f}x")
    print("\n주: 확장노드가 줄어도 J 가 1.000x 를 넘으면 그 h 는 비최적해를 낸 것 (G3-N1 근거자료).")
else:
    print("건너뜀")

## 13. 평가 — success_rate / length_ratio

- `success_rate` = 목표 도달 비율 (고정 평가 시나리오 48개, 결정적 정책)
- `length_ratio` = 에이전트 경로길이(m) / **k 제약 A\* 최적 경로길이(m)** — 1.0 이 최적
- `j_ratio` = 에이전트 J / A\* 최적 J — 엘보까지 포함한 **진짜 목적함수 비교**
- `deadlock_rate` = 유효 이동이 0개가 된 비율 (V1 이 말한 2% 사례가 실제로 0 인지 확인)

기준선은 시나리오 **방향별로** 저장돼 있어 평가 시 에이전트와 A* 가 같은 초기 상태에서 출발한다.

In [ ]:
#@title 13. 평가 함수
def evaluate_routing(model, pool, deterministic: bool = True, seed: int = 0) -> dict:
    env = PipeRoutingEnvStep1(pool, eval_mode=True, seed=seed)
    rng = np.random.default_rng(seed)
    rows = []
    for i in range(len(pool)):
        obs, _ = env.reset(options={"index": i})
        done = False
        info = {}
        while not done:
            m = env.action_masks()
            if model is None:
                a = int(rng.choice(np.where(m)[0]))            # 랜덤 기준선
            else:
                a, _ = model.predict(obs, action_masks=m, deterministic=deterministic)
                a = int(a)
            obs, r, term, trunc, info = env.step(a)
            done = term or trunc
        rows.append(info)

    succ = np.array([r["success"] for r in rows])
    lr = np.array([r["length_ratio"] for r in rows], dtype=float)
    jr = np.array([r["j_ratio"] for r in rows], dtype=float)
    nb = np.array([r["n_bends"] for r in rows], dtype=float)
    nb0 = np.array([r["bends_base"] for r in rows], dtype=float)
    m_ok = succ > 0.5
    f = lambda v: float(np.mean(v)) if len(v) else float("nan")
    return dict(
        success_rate=float(succ.mean()),
        deadlock_rate=float(np.mean([r["deadlock"] for r in rows])),
        timeout_rate=float(np.mean([r["timeout"] for r in rows])),
        length_ratio=f(lr[m_ok]),
        j_ratio=f(jr[m_ok]),
        bends_agent=f(nb[m_ok]),
        bends_astar=f(nb0[m_ok]),
        ep_steps=float(np.mean([r["ep_steps"] for r in rows])),
        n_invalid=float(np.mean([r["n_invalid"] for r in rows])),
    )

def fmt_metrics(m: dict) -> str:
    return (f"success {m['success_rate']:.3f} | length_ratio {m['length_ratio']:.3f} | "
            f"j_ratio {m['j_ratio']:.3f} | bends {m['bends_agent']:.1f} vs A* {m['bends_astar']:.1f} | "
            f"deadlock {m['deadlock_rate']:.3f} | timeout {m['timeout_rate']:.3f} | "
            f"steps {m['ep_steps']:.0f}"
            + (f" || train success {m['train_success']:.3f}" if "train_success" in m else ""))

# 랜덤(마스킹된) 정책 기준선 — 학습이 무엇을 이겨야 하는지의 바닥값
_rand = evaluate_routing(None, EVAL_POOL)
print("랜덤 정책            :", fmt_metrics(_rand))

# Layer 0 확장 옵션의 효과 측정: 1-ply 데드락 회피 마스크를 켜면 데드락이 얼마나 줄어드나
_prev = cfg.deadlock_1ply
cfg.deadlock_1ply = not _prev
print(f"랜덤 + 1ply={cfg.deadlock_1ply}   :", fmt_metrics(evaluate_routing(None, EVAL_POOL)))
cfg.deadlock_1ply = _prev
print(f"\n현재 설정: deadlock_1ply={cfg.deadlock_1ply}")
print("주: 랜덤 정책의 deadlock_rate 가 1.0 이어도 정상이다. k=6 마스크 하에서 무작정 직진하면"
      " 벽 앞에서 s<k 라 꺾지 못해 막힌다. 학습은 '미리 꺾는' 정책을 찾는 것이고,"
      " A* 가 전 시나리오를 푼다는 사실이 해가 존재함을 보증한다 (§7 도달성 검증).")

## 14. wandb + 콜백

API key 가 없으면 자동으로 `offline` 로 떨어진다 (Colab 에서 `wandb.login()` 먼저 실행 권장).

In [ ]:
#@title 14. wandb 초기화 + 평가 콜백
USE_WANDB = True   #@param {type:"boolean"}

from stable_baselines3.common.callbacks import BaseCallback

wandb = None
WandbCallback = None
run = None
if USE_WANDB:
    try:
        import wandb
        from wandb.integration.sb3 import WandbCallback
    except Exception as e:
        print("wandb import 실패 → 로깅 비활성:", e); USE_WANDB = False

if USE_WANDB:
    mode = cfg.wandb_mode or ("online" if os.environ.get("WANDB_API_KEY") else None)
    if mode is None:
        try:
            mode = "online" if wandb.login(timeout=15) else "offline"
        except Exception:
            mode = "offline"
    run = wandb.init(
        project=cfg.wandb_project,
        name=cfg.wandb_run_name or f"G3-step1-{cfg.nominal}-{cfg.phi_type}-{time.strftime('%m%d-%H%M')}",
        config={**asdict(cfg), "spec": asdict(SPEC), "DIAG_J": DIAG_J, "k": K,
                "obs_dim": OBS_DIM, "n_actions": N_ACTIONS,
                "layer0": ["collision", "boundary", "reverse(OPP26)", "bend_radius(lookahead k)"],
                "notes": ["G3-N1 h_clip", "G3-N2 max_turn_deg", "G3-N3 terminal Phi"]},
        sync_tensorboard=True, mode=mode, save_code=False,
    )
    print("wandb:", mode, "|", getattr(run, "url", "(offline)"))

HISTORY = []

class G3EvalCallback(BaseCallback):
    """cfg.eval_every 스텝마다 고정 평가셋으로 success_rate / length_ratio 측정."""
    def __init__(self, pool, every: int, verbose: int = 1):
        super().__init__(verbose)
        self.pool, self.every = pool, every
        self._last = 0

    def _log(self, m):
        m = dict(m)
        m["timesteps"] = self.num_timesteps
        if self.model.ep_info_buffer:
            m["train_ep_rew_mean"] = float(np.mean([e["r"] for e in self.model.ep_info_buffer]))
            m["train_ep_len_mean"] = float(np.mean([e["l"] for e in self.model.ep_info_buffer]))
            # 학습 중(확률적 정책) 성공률. 평가(결정적)와 크게 다르면 탐색/결정성 문제다.
            for key in ("success", "deadlock"):
                vals = [e[key] for e in self.model.ep_info_buffer if key in e]
                if vals:
                    m[f"train_{key}"] = float(np.mean(vals))
        HISTORY.append(m)
        for k_, v in m.items():
            if k_ != "timesteps":
                self.logger.record(f"eval/{k_}", v)
        if USE_WANDB and run is not None:
            wandb.log({f"eval/{k_}": v for k_, v in m.items() if k_ != "timesteps"},
                      step=self.num_timesteps)
        if self.verbose:
            print(f"[{self.num_timesteps:>7,}] {fmt_metrics(m)}")

    def _on_step(self) -> bool:
        if self.num_timesteps - self._last >= self.every:
            self._last = self.num_timesteps
            self._log(evaluate_routing(self.model, self.pool))
        return True

    def _on_training_end(self) -> None:
        self._log(evaluate_routing(self.model, self.pool))

## 15. 학습 — MaskablePPO (CLAUDE.md §0 변경불가 항목)

`sb3-contrib MaskablePPO` 고정. rollout buffer / advantage / clip loss 는 **원본 그대로**,
개입 지점은 `policy_kwargs.features_extractor_class` 하나뿐이다 (G2 통과 조건).

소요시간: **CPU 8 envs 실측 42분**. T4 는 미실측 (추측: 정책이 작은 MLP 라 환경 롤아웃이 병목이므로 비슷).
급하면 `cfg.total_timesteps` 를 줄이되, **400k 에서는 success ≈ 0.58 로 아직 수렴 전이다** (셀 19 실측).

In [ ]:
#@title 15. 학습
from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker
from sb3_contrib.common.maskable.utils import is_masking_supported
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.utils import set_random_seed

set_random_seed(SEED)

TB_LOG = None
try:
    import tensorboard  # noqa: F401
    TB_LOG = "./tb"
except Exception:
    print("tensorboard 미설치 → TB 로깅 비활성 (wandb 는 그대로 기록됨)")

def mask_fn(env):
    return env.action_masks()

def make_env(rank: int):
    def _init():
        e = PipeRoutingEnvStep1(TRAIN_POOL, eval_mode=False, seed=SEED + rank)
        e = ActionMasker(e, mask_fn)
        return Monitor(e, info_keywords=INFO_KEYS)
    return _init

venv = DummyVecEnv([make_env(i) for i in range(cfg.n_envs)])
assert is_masking_supported(venv), "action mask 미지원 — Layer 0 가 동작하지 않는다"

policy_kwargs = dict(
    features_extractor_class=GNARLFeaturesExtractor,
    features_extractor_kwargs=dict(features_dim=cfg.features_dim, hidden=cfg.hidden,
                                   clip_mode=cfg.h_clip_mode),
    net_arch=dict(pi=[128, 128], vf=[128, 128]),
)

model = MaskablePPO(
    "MlpPolicy", venv,
    learning_rate=cfg.learning_rate,
    n_steps=cfg.n_steps, batch_size=cfg.batch_size, n_epochs=cfg.n_epochs,
    gamma=cfg.gamma,                      # §9 조건 2: γ_PBS = γ_PPO
    clip_range=cfg.clip_range, ent_coef=cfg.ent_coef,
    policy_kwargs=policy_kwargs, tensorboard_log=TB_LOG, seed=SEED, verbose=0,
    device="auto",
)
assert abs(model.gamma - cfg.gamma) < 1e-12, "§9 조건 2 위반 (γ_PBS ≠ γ_PPO)"

# 사전학습된 h(n) 주입 (공유 extractor 별칭까지 전부)
if H_EXTRACTOR_SD is not None:
    loaded = 0
    for mod_ in model.policy.modules():
        if isinstance(mod_, GNARLFeaturesExtractor):
            mod_.h_net.load_state_dict(H_EXTRACTOR_SD); loaded += 1
    print(f"사전학습 h(n) 주입: {loaded} 개 extractor")

print(model.policy)
print(f"\ndevice={model.device} | {cfg.n_envs} envs × n_steps {cfg.n_steps} = "
      f"rollout {cfg.n_envs*cfg.n_steps} | 총 {cfg.total_timesteps:,} steps")

callbacks = [G3EvalCallback(EVAL_POOL, cfg.eval_every)]
if USE_WANDB and WandbCallback is not None and run is not None:
    callbacks.append(WandbCallback(gradient_save_freq=0, verbose=0))

t0 = time.time()
import importlib.util as _ilu
_progress = _ilu.find_spec("rich") is not None and _ilu.find_spec("tqdm") is not None
model.learn(total_timesteps=cfg.total_timesteps, callback=callbacks, progress_bar=_progress)
print(f"\n학습 완료: {time.time()-t0:.0f}s ({(time.time()-t0)/60:.1f}분)")
model.save("g3_step1_maskableppo")

## 16. 결과 / G3 판정

**판정 기준 (추측: 목표치 — 실측 후 CLAUDE.md §15 에 기록할 것)**

| 항목 | 기준 |
|---|---|
| success_rate | ≥ 0.95 (최근 3회 평가 평균) |
| length_ratio | ≤ 1.10 (k 제약 A* 대비) |
| deadlock_rate | = 0 (lookahead k 가 제대로 동작하면 0 이어야 함 — V1) |
| 수렴 | success_rate 가 마지막 3회에서 단조 하락하지 않음 |

In [ ]:
#@title 16. 곡선 + 판정
import matplotlib.pyplot as plt
# 그래프 라벨은 ASCII 로 쓴다 (Colab 기본 matplotlib 에 한글 폰트가 없어 □ 로 깨짐)

ts = [h["timesteps"] for h in HISTORY]
fig, ax = plt.subplots(2, 2, figsize=(12, 7))
ax[0,0].plot(ts, [h["success_rate"] for h in HISTORY], marker="o")
ax[0,0].axhline(cfg.target_success_rate, ls="--", c="r"); ax[0,0].set_title("success_rate"); ax[0,0].set_ylim(0, 1.02)
ax[0,1].plot(ts, [h["length_ratio"] for h in HISTORY], marker="o", label="length_ratio")
ax[0,1].plot(ts, [h["j_ratio"] for h in HISTORY], marker="s", label="j_ratio (incl. elbows)")
ax[0,1].axhline(cfg.target_length_ratio, ls="--", c="r"); ax[0,1].axhline(1.0, ls=":", c="k")
ax[0,1].set_title("ratio vs k-constrained A* optimum"); ax[0,1].legend()
ax[1,0].plot(ts, [h.get("train_ep_rew_mean", np.nan) for h in HISTORY], marker="o")
ax[1,0].set_title("train ep_rew_mean (-dJ + PBS shaping)")
ax[1,1].plot(ts, [h["deadlock_rate"] for h in HISTORY], marker="o", label="deadlock")
ax[1,1].plot(ts, [h["timeout_rate"] for h in HISTORY], marker="s", label="timeout")
ax[1,1].set_title("failure modes"); ax[1,1].legend(); ax[1,1].set_ylim(-0.02, 1.02)
for a in ax.ravel(): a.set_xlabel("timesteps"); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig("g3_curves.png", dpi=120); plt.show()

final = evaluate_routing(model, EVAL_POOL)
tail = HISTORY[-3:] if len(HISTORY) >= 3 else HISTORY
sr_tail = float(np.mean([h["success_rate"] for h in tail]))
lr_tail = float(np.nanmean([h["length_ratio"] for h in tail]))
sr_seq = [h["success_rate"] for h in tail]
checks = {
    f"success_rate ≥ {cfg.target_success_rate}": sr_tail >= cfg.target_success_rate,
    f"length_ratio ≤ {cfg.target_length_ratio}": lr_tail <= cfg.target_length_ratio,
    "deadlock_rate = 0": final["deadlock_rate"] == 0.0,
    "수렴 (마지막 3회 단조하락 아님)": not (len(sr_seq) == 3 and sr_seq[0] > sr_seq[1] > sr_seq[2]),
}
print("최종 평가:", fmt_metrics(final))
print(f"최근 3회 평균: success {sr_tail:.3f} | length_ratio {lr_tail:.3f}\n")
for k_, v in checks.items():
    print(("  PASS  " if v else "  FAIL  ") + k_)
G3_PASS = all(checks.values())
print("\n" + ("G3 통과 — Step 1 정식 학습 진행 가능 (§14-3)"
              if G3_PASS else "G3 미통과 — 아래 '실패 시 점검 순서' 참고"))

### 경로 눈으로 확인

수치만으로는 "왜 J 가 큰지"가 안 보인다. 에이전트 경로와 k 제약 A* 최적경로를 같이 그린다.
엘보 1개 = 직진 4.7칸(100A)이므로, 길이는 비슷한데 J 가 크면 **불필요한 꺾임**이 원인이다.

In [ ]:
#@title 16b. 경로 3D 비교 (에이전트 vs A*)
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

def rollout_path(model, pool, idx, deterministic=True):
    env = PipeRoutingEnvStep1(pool, eval_mode=True)
    obs, _ = env.reset(options={"index": idx})
    done = False; info = {}
    while not done:
        m = env.action_masks()
        a, _ = model.predict(obs, action_masks=m, deterministic=deterministic)
        obs, r, te, tr, info = env.step(int(a))
        done = te or tr
    return env.path, info, env.base

def plot_routes(model, pool, indices=(0, 1, 2)):
    fig = plt.figure(figsize=(5 * len(indices), 4.6))
    for j, idx in enumerate(indices):
        sc = pool[idx]
        path, info, base = rollout_path(model, pool, idx)
        ax = fig.add_subplot(1, len(indices), j + 1, projection="3d")
        ob = np.argwhere(~sc["free"])
        if len(ob):
            sub = ob[:: max(1, len(ob) // 600)]
            ax.scatter(sub[:, 0], sub[:, 1], sub[:, 2], s=4, c="0.75", alpha=.25, marker="s")
        ap = np.array([st[0] for st in base["states"]])
        ax.plot(ap[:, 0], ap[:, 1], ap[:, 2], c="tab:green", lw=2, ls="--", label=f"A* J={base['J']:.1f}")
        pp = np.array(path)
        ax.plot(pp[:, 0], pp[:, 1], pp[:, 2], c="tab:blue", lw=2,
                label=f"agent J={info['j_agent']:.1f}")
        ax.scatter(*sc["start"], c="k", s=45, marker="o")
        ax.scatter(*sc["goal"], c="r", s=60, marker="*")
        ax.set_xlim(0, cfg.NX); ax.set_ylim(0, cfg.NY); ax.set_zlim(0, cfg.NZ)
        ax.set_title(f"#{idx} " + ("OK" if info["success"] > 0.5 else
                                   ("DEADLOCK" if info["deadlock"] > 0.5 else "TIMEOUT")), fontsize=10)
        ax.legend(fontsize=7, loc="upper left")
    plt.tight_layout(); plt.savefig("g3_routes.png", dpi=120); plt.show()

plot_routes(model, EVAL_POOL, indices=(0, 1, 2))

In [ ]:
#@title 17. 저장 + wandb 마무리
summary = dict(
    g3_pass=bool(G3_PASS), final=final, tail_success=sr_tail, tail_length_ratio=lr_tail,
    history=HISTORY, config=asdict(cfg), spec=asdict(SPEC), DIAG_J=DIAG_J, k=K,
    random_baseline=_rand,
    checklist=_res,
)
with open("g3_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2, default=float)
print("저장: g3_step1_maskableppo.zip / g3_summary.json / g3_curves.png / g3_routes.png / g3_scenarios.pkl")

if USE_WANDB and run is not None:
    wandb.log({f"final/{k_}": v for k_, v in final.items()})
    wandb.run.summary["g3_pass"] = bool(G3_PASS)
    wandb.run.summary["tail_success_rate"] = sr_tail
    wandb.run.summary["tail_length_ratio"] = lr_tail
    try:
        for _f in ("g3_summary.json", "g3_curves.png", "g3_routes.png"):
            wandb.save(_f)
    except Exception as e:
        print("wandb.save 건너뜀:", e)
    wandb.finish()

## 18. V4-3 — 맨해튼 vs 복합 Φ 수렴속도 비교 (§9, §14-2)

CLAUDE.md §9 는 "V4-3 수렴 속도 실측은 G3(Colab)에서 확정" 이라고 못박았다.
같은 시드/시나리오 풀로 `phi_type` 만 바꿔 두 번 돌린 뒤 **같은 success_rate 도달에 필요한 timesteps**
를 비교한다. 셀 18 은 그 실행 틀이다 (기본 OFF — 학습시간 2배).

> Φ 후보는 §9 표의 A(맨해튼) / E(복합 B×0.7+D×0.3) 을 쓴다.
> 복합 Φ 도 `Phi()` 안에서 `pos==goal → 0.0` 으로 **종단 조건을 명시**하고 있다.
> 혼잡도 항은 목표칸에서 0 이 아니므로, 이 명시가 없으면 §9 조건 1 을 위반한다.

In [ ]:
#@title 18. (선택) Φ ablation — phi_type 만 바꿔 재학습
RUN_PHI_ABLATION = False   #@param {type:"boolean"}

def train_variant(phi_type: str, goal_ratio=0.7, cong_ratio=0.3, steps=None, tag=""):
    global HISTORY
    old = (cfg.phi_type, cfg.phi_goal_ratio, cfg.phi_cong_ratio)
    cfg.phi_type, cfg.phi_goal_ratio, cfg.phi_cong_ratio = phi_type, goal_ratio, cong_ratio
    hist_backup, HISTORY = HISTORY, []
    try:
        v = DummyVecEnv([make_env(100 + i) for i in range(cfg.n_envs)])
        m = MaskablePPO("MlpPolicy", v, learning_rate=cfg.learning_rate, n_steps=cfg.n_steps,
                        batch_size=cfg.batch_size, n_epochs=cfg.n_epochs, gamma=cfg.gamma,
                        clip_range=cfg.clip_range, ent_coef=cfg.ent_coef,
                        policy_kwargs=policy_kwargs, seed=SEED, verbose=0)
        if H_EXTRACTOR_SD is not None:
            for mod_ in m.policy.modules():
                if isinstance(mod_, GNARLFeaturesExtractor):
                    mod_.h_net.load_state_dict(H_EXTRACTOR_SD)
        m.learn(total_timesteps=steps or cfg.total_timesteps,
                callback=[G3EvalCallback(EVAL_POOL, cfg.eval_every)], progress_bar=_progress)
        out = list(HISTORY)
    finally:
        HISTORY = hist_backup
        cfg.phi_type, cfg.phi_goal_ratio, cfg.phi_cong_ratio = old
    return out

def steps_to_reach(hist, thr=0.9):
    for h in hist:
        if h["success_rate"] >= thr:
            return h["timesteps"]
    return None

if RUN_PHI_ABLATION:
    h_man = HISTORY if cfg.phi_type == "manhattan" else train_variant("manhattan")
    h_cmb = train_variant("combined", 0.7, 0.3)
    for nm, h in (("manhattan", h_man), ("combined", h_cmb)):
        print(f"{nm:10s} success 0.9 도달 = {steps_to_reach(h)} steps | "
              f"최종 success {h[-1]['success_rate']:.3f} length_ratio {h[-1]['length_ratio']:.3f}")
    plt.figure(figsize=(7, 4))
    for nm, h in (("manhattan", h_man), ("combined", h_cmb)):
        plt.plot([x["timesteps"] for x in h], [x["success_rate"] for x in h], marker="o", label=nm)
    plt.xlabel("timesteps"); plt.ylabel("success_rate"); plt.legend(); plt.grid(alpha=.3)
    plt.title("V4-3 convergence speed (Phi candidates)"); plt.savefig("g3_v43_phi.png", dpi=120); plt.show()
else:
    print("건너뜀 (RUN_PHI_ABLATION=False)")

---

## 19. 이 노트북을 만들며 돌린 실측 기록

CLAUDE.md 의 규칙대로 **추측과 실측을 구분**해 남긴다.
아래는 전부 CPU(8 envs)에서 돌린 값이라 절대치는 T4 와 다를 수 있지만, **상대 비교는 유효하다.**
"8레이아웃" = 학습 풀과 평가 풀을 같은 시나리오 8개로 둔 암기 실험 (일반화 말고 최적화 능력만 본다).

### 19-1. 턴 각도 상한이 데드락의 주범이었다 (가장 큰 발견)

초안은 109°/120°/135° 급선회가 배관상 비현실적이라 `max_turn_deg=90` 으로 막았다 (CLAUDE.md 에 없는 제약).
그 결과가 데드락 대유행이었다.

| `max_turn_deg` | 랜덤 정책 success | 150k 학습 후 success | deadlock |
|---|---|---|---|
| 90 (초안, CLAUDE.md 밖) | 0.000 | 0.500 | 0.500 |
| **180 (CLAUDE.md 원문)** | **1.000** | **0.875** | **0.125** |

이유: 벤딩반경 마스크는 "꺾으려면 s≥k **그리고** 새 방향으로 k칸 여유" 를 요구한다.
여기에 90° 상한까지 겹치면 벽 앞에서 빠져나갈 각도가 남지 않는다. **급선회가 사실상 유일한 탈출구였다.**

→ CLAUDE.md 원문(역방향만 금지)이 옳다. 기본값을 180 으로 되돌렸다.
→ 대신 급선회를 **Layer 1 에서 제대로 청구**한다: 편향각 >112.5° 는 엘보 하나로 못 꺾으므로 `90° + 45°` 로 계산.
   제약(Layer 0)을 추가하는 대신 목적함수(Layer 1)로 다루는 쪽이 3층 구조에 맞다.

### 19-2. 데드락 종단 비용 (8레이아웃, 150k, `ent_coef=0.05`)

| `fail_cost_mode` | success |
|---|---|
| `none` | 0.125 |
| `h_star` | 0.250 |
| **`const=2.0`** | **0.500** |

`fail_cost` 가 없으면 **즉시 포기가 국소최적**이다 (셀 8 표 참조). `h_star` 는 하한이라 약하다.

### 19-3. `ent_coef` (8레이아웃, 150k, `h_star`)

0.01 → 0.05 로 올리면 success 0.125 → 0.250. 데드락 지형에서 정책이 일찍 결정화되는 것을 늦춘다.
CLAUDE.md §10 의 autoresearch 탐색 대상에는 없지만, **Step 1 의 기본값으로는 0.05 가 필요했다.**

### 19-4. 효과가 없었던 것 (기록용 — 재시도 방지)

- **26방향 직진여유 채널 추가** (obs 50→76): 풀스케일 400k 에서 success 0.02~0.06 → 0.02~0.06 으로 **차이 없음**.
  당시 병목은 19-1 이었다. 채널 자체는 마스크와 같은 정보를 정책에 주므로 유지했지만 **효과는 미입증**이다.
- **1-ply 데드락 회피 마스크**: 데드락율 0.80 → 0.78, 마스크가 실제로 달라진 스텝 1.4%.
  데드락은 2수 이상 앞에서 결정된다. 기본 off.

### 19-5. 풀스케일 (192 학습 / **48 미학습** 평가, 기본 설정)

2M 스텝 전체 실행 (**CPU 8 envs 에서 42분**. T4 소요시간은 미실측 —
추측: 정책이 작은 MLP 라 GPU 보다 환경 롤아웃이 병목이므로 비슷할 것이다):

| timesteps | success | length_ratio | deadlock | timeout | train success |
|---|---|---|---|---|---|
| 랜덤 정책 | 0.146 | 4.88 | 0.438 | 0.417 | — |
| 100k | 0.104 | 3.55 | 0.500 | — | 0.13 |
| 300k | 0.500 | 3.58 | 0.354 | — | 0.46 |
| 600k | 0.583 | 2.65 | 0.292 | 0.125 | 0.60 |
| 900k | 0.583 | 3.24 | 0.229 | 0.188 | 0.61 |
| 1.2M | 0.479 | 3.31 | 0.333 | 0.188 | 0.62 |
| 1.5M | 0.604 | 2.75 | 0.271 | 0.125 | 0.77 |
| 1.8M | 0.500 | 2.60 | 0.229 | 0.271 | 0.63 |
| **2.0M** | **0.604** | **3.36** | **0.229** | 0.167 | **0.74** |

**해석 (추측 아님, 위 수치에서 바로 읽히는 것):**
- 학습은 분명히 된다: 랜덤 0.146 → 0.60, 데드락 0.44 → 0.23.
- 그러나 **300k 이후 success 0.5~0.6 에서 평평하다.** 1.7M 스텝을 더 써도 올라가지 않았다.
- `length_ratio ≈ 2.6~3.4` 로 고착. 즉 **멀리 돌아가는 안전한 경로**를 학습했다.
  원인은 목적함수 균형이다: `fail_cost=2.0` > 40칸 우회 비용(≈1.3) 이라 **실패를 피하는 게 우회보다 싸다.**
  목적함수대로 행동하는 것이므로 버그가 아니다 — 상수를 바꾸면 균형점이 바뀐다.
- 초반에는 train ≈ eval 이었는데(0.61 vs 0.58 @900k) 후반에 벌어진다(**0.74 vs 0.60 @2M**).
  → 초반 정체는 최적화 문제, 후반 정체는 **일반화 문제**다. 둘의 처방이 다르다
  (전자는 목적함수/탐색, 후자는 `n_train_scen` 증가).
- 실패의 절반 이상이 이제 **timeout(0.17)** 이다. 데드락은 잡혔고 "제 시간에 도달"이 남은 과제다.

### 19-6. 그래서 G3 는?

**CPU 2M 스텝 기준: success 0.60 / length_ratio 3.36 — §16 목표치(0.95 / 1.10) 미달.**
"구조는 돌아간다, 수렴은 아직" 이 현재까지의 G3 실측 결과다.

무엇이 검증됐고 무엇이 안 됐는지 구분해 둔다.

| 항목 | 상태 |
|---|---|
| G2 통합구조가 실규모에서 도는가 (feature extractor / mask / rollout) | ✅ 400k~2M 스텝 정상 동작 |
| §11 체크리스트 10항목 | ✅ 전부 통과 |
| h(n) 사전학습이 휴리스틱보다 나은가 | ✅ MAE 2.35 kg vs 맨해튼 3.18 / octile3d 4.90 |
| 학습이 되긴 하는가 | ✅ 랜덤 0.146 → 0.604 |
| **§16 목표치 수렴** | ❌ **미달 (0.60 / 3.36)** |
| V4-3 (맨해튼 vs 복합 Φ) | 🔜 셀 18 `RUN_PHI_ABLATION=True` 로 실행 |

다음에 돌릴 순서 (효과가 클 것 같은 순):

1. **`fail_cost_const` sweep (1.2 / 2.0 / 3.0)** — 성공률 ↔ 경로품질 트레이드오프의 핵심 손잡이.
   현재 2.0 은 성공률 쪽으로 치우쳐 있다 (그래서 `length_ratio ≈ 3`).
2. **`n_train_scen` 증가 (192 → 500+)** — 2M 시점의 train 0.74 vs eval 0.60 격차가 일반화 부족을 가리킨다.
   시나리오 생성은 A* 도달성 검증 때문에 느리므로 캐시를 쓸 것.
3. `learning_rate` / `n_steps` 는 §10 탐색범위 `(1e-4, 5e-4)` / `(256, 2048)` 안에서만 건드린다.
4. 커리큘럼: `min_path_cells` 를 낮게 시작해 올린다 (A* 는 도달성 검증용일 뿐 정답 시연이 아니므로 §1 과 충돌하지 않는다).
5. 그래도 정체하면 **환경 난이도 자체를 기록**할 것: 20×20×10 에 k=6 은 높이 500mm 안에서
   직진 300mm 를 요구하는 배치다. **수직 방향으로는 사실상 벤딩 1회가 한계**다.
   이건 알고리즘 문제가 아니라 **격자/구경 조합의 물리적 제약**이므로 CLAUDE.md §13 에 기록할 가치가 있다.
   (확인 방법: `cfg.NZ` 를 20 으로 올리거나 `cfg.nominal="25A"` (k=2) 로 낮춰 같은 학습을 돌려 비교.)

---

## 20. 실패 시 점검 순서 (G3 미통과일 때)

1. **success_rate 가 0 근처에서 안 움직임**
   → `HISTORY` 의 `timeout_rate` 확인. 1.0 이면 탐색 실패다.
   `cfg.max_steps` 를 늘리기 전에 **Φ 부호**부터 확인 (Φ 는 항상 ≤ 0, 목표에서 0).
2. **deadlock_rate > 0**
   → lookahead 가 안 걸린 것. §5 의 `is_valid_turn` 2번 조건(새 방향 k칸 비어있나)을 확인.
   V1 은 lookahead 없으면 도달가능 상태의 2% 에서 데드락이라고 실측했다.
3. **success 는 높은데 length_ratio 가 1.3+**
   → J 정규화 스케일(`J_MAX`) 대비 엘보 비중을 확인.
   100A 는 엘보 1개 = 직진 4.7칸이라 J 에서 엘보가 지배적이다. `j_ratio` 와 `bends_agent` 를 같이 볼 것.
4. **학습 초반에 발산**
   → `cfg.learning_rate` 를 §10 탐색범위 `(1e-4, 5e-4)` 안에서 낮춘다. 그 밖은 탐색 대상이 아니다.
5. **h(n) 사전학습 MAE 가 맨해튼보다 나쁨**
   → §6 G1-Q2 의 J 정규화 문제. `DIAG_J` 와 `build_obs` 의 `obs[33]` 정규화를 확인.

## 21. G3 종료 후 CLAUDE.md 갱신 항목

- §13 Lock `G3` → 결과 기록 (통과/미통과 + 수치)
- §9 V4-3 → `RUN_PHI_ABLATION` 결과로 맨해튼/복합 확정
- §15 검증 결과 요약에 G3 행 갱신
- 새로 생긴 미결 3건을 §13 에 추가할지 판단:
  - **G3-N1** `min(h_learned, h_manhattan)` 은 26-연결에서 admissibility 를 보장하지 않음 (맨해튼 = 과대추정).
    엄밀 하한은 octile3d = `(a-b) + √2(b-c) + √3c`. 셀 12 가 근거 수치를 만든다.
  - **G3-N2** 턴 각도 상한 (기본 90°). CLAUDE.md 는 180°(역방향)만 금지한다.
    109°/120°/135° 급선회를 허용할지 = 배관 설계 판단 필요.
  - **G3-N3** 종단 처리. Φ(종단)=0 은 **목표 도달 종단에만** 적용해야 한다
    (데드락/타임아웃에 적용하면 실패에 보너스가 붙어 §9 probe4 위반).

*이 노트북은 CLAUDE.md 만 보고 작성했다. 추측은 "추측:" / 미결은 "미결:" 로 표기했다.*